In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql.functions import lit, col, when

# Initialize SparkSession
spark = SparkSession.builder.getOrCreate()

# Define all layers and tables
superstore_tables = {
    "bronze": [
        "superstore_catalog.superstore_bronze.superstore_raw",
        "superstore_catalog.superstore_bronze.orders",
        "superstore_catalog.superstore_bronze.customers",
        "superstore_catalog.superstore_bronze.products",
        "superstore_catalog.superstore_bronze.sales"
    ],
    "silver": [
        "superstore_catalog.superstore_silver.orders",
        "superstore_catalog.superstore_silver.customers",
        "superstore_catalog.superstore_silver.products",
        "superstore_catalog.superstore_silver.sales"
    ],
    "quarantine": [
        "superstore_catalog.superstore_quarantine.orders_dirty",
        "superstore_catalog.superstore_quarantine.customers_dirty",
        "superstore_catalog.superstore_quarantine.products_dirty",
        "superstore_catalog.superstore_quarantine.sales_dirty"
    ],
    "audit": [
        "superstore_catalog.superstore_audit.orders_duplicates",
        "superstore_catalog.superstore_audit.customers_duplicates",
        "superstore_catalog.superstore_audit.products_duplicates",
        "superstore_catalog.superstore_audit.sales_duplicates"
    ],
    "gold": [
        "superstore_catalog.superstore_gold.facts_orders",
        "superstore_catalog.superstore_gold.dim_customers",
        "superstore_catalog.superstore_gold.dim_products",
        "superstore_catalog.superstore_gold.facts_sales"
    ],
}

# Function to count rows safely
def count_rows(table_name):
    if spark.catalog.tableExists(table_name):
        return spark.table(table_name).count()
    else:
        return 0

# Collect results in a list
counts_list = []

for layer, tables in superstore_tables.items():
    for tbl in tables:
        counts_list.append(
            Row(layer=layer, table_name=tbl, row_count=count_rows(tbl))
        )

# Convert to DataFrame for display
counts_df = spark.createDataFrame(counts_list)

# Sort the DataFrame by the custom order of layers and then by table name
sorted_counts_df = counts_df.orderBy(
    when(col("layer") == "bronze", lit(1))
    .when(col("layer") == "silver", lit(2))
    .when(col("layer") == "gold", lit(3))
    .when(col("layer") == "audit", lit(4))
    .when(col("layer") == "quarantine", lit(5))
    .otherwise(lit(6)),  # Default if something goes wrong
    "table_name"  # Then sort by table name alphabetically
)

# Display the sorted DataFrame (without including the custom "layer_order" column)
sorted_counts_df.display()

